<a href="https://colab.research.google.com/github/ZacharyJimmy/DataMining/blob/main/DataMiningCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Import Libs
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split


from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import sklearn.metrics as metrics

In [ ]:
#Load Dataset

df = pd.read_csv('DailyDataset.csv')

In [ ]:
#Convert Date

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

In [ ]:
#Convert Prices to Returns
currency_cols = ['USD','EUR','JPY','GBP','CAD','CHF','INR','CNY','TRY',
                 'SAR','IDR','AED','THB','VND','EGP','KRW','RUB','ZAR','AUD']

# Convert currency columns to numeric, removing commas first
for col in currency_cols:
    df[col] = df[col].astype(str).str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

returns = df[currency_cols].pct_change().dropna()

In [ ]:
#Standardize Returns
scaler = StandardScaler()
returns_scaled = scaler.fit_transform(returns)

In [ ]:
#Correlation Analysis
corr_matrix = pd.DataFrame(returns_scaled, columns=currency_cols).corr()
corr_matrix

In [ ]:
#Visualize Correl
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, cmap='coolwarm')
plt.show()

In [ ]:
#PCA
pca = PCA()
components = pca.fit_transform(returns_scaled)

explained_variance = pca.explained_variance_ratio_
print(explained_variance)

In [ ]:
#Examine PCA
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(currency_cols))],
    index=currency_cols
)

loadings.head()

In [ ]:
#Plot First Principal Component Over Time
pc1_series = components[:, 0]

plt.figure(figsize=(12,5))
plt.plot(df['Date'].iloc[1:], pc1_series)
plt.title("Global Gold Movement (PC1)")
plt.show()

In [ ]:
#----------PHASE 3------------------

In [ ]:
# STEP 1: SELECT NUMBER OF PCs

cum_var = np.cumsum(pca.explained_variance_ratio_)

k = np.argmax(cum_var >= 0.90) + 1

print(f"Selected number of PCs: {k}")
print(f"Cumulative explained variance: {cum_var[k-1]:.4f}")

In [ ]:
# STEP 2: CREATE PCA DATASET

PC_df = pd.DataFrame(
    components[:, :k],
    columns=[f'PC{i+1}' for i in range(k)]
)

In [ ]:
# STEP 3: FEATURE ENGINEERING

# Lag features
for i in range(1, k+1):
    PC_df[f'PC{i}_lag1'] = PC_df[f'PC{i}'].shift(1)
    PC_df[f'PC{i}_lag7'] = PC_df[f'PC{i}'].shift(7)

# Rolling features
PC_df['PC1_MA7'] = PC_df['PC1'].rolling(7).mean()
PC_df['PC1_VOL7'] = PC_df['PC1'].rolling(7).std()

# Drop NaN
PC_df = PC_df.dropna().reset_index(drop=True)

In [ ]:
# STEP 4: CREATE X & y


X = PC_df.drop(columns=['PC1'])
y = PC_df['PC1']

print("Final Dataset Shape:", PC_df.shape)

In [ ]:
# STEP 5: TRAIN TEST SPLIT (TIME SERIES)

split = int(len(PC_df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [ ]:
# STEP 6: MODEL TRAINING

lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=300, random_state=42)
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42)

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

In [ ]:
# STEP 7: MODEL EVALUATION

def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name}: RMSE = {rmse:.4f}, R² = {r2:.4f}")

pred_lr = lr.predict(X_test)
pred_rf = rf.predict(X_test)
pred_gb = gb.predict(X_test)

evaluate(y_test, pred_lr, "Linear Regression")
evaluate(y_test, pred_rf, "Random Forest")
evaluate(y_test, pred_gb, "Gradient Boosting")

In [ ]:
#MAE MSE RMSE

def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(metrics.mean_squared_error(y_true, y_pred))
    r2 = metrics.r2_score(y_true, y_pred)
    mae = metrics.mean_absolute_error(y_true, y_pred)
    mse = metrics.mean_squared_error(y_true, y_pred)
    print(f"{model_name}: RMSE = {rmse:.4f}, R² = {r2:.4f}, MAE = {mae:.4f}, MSE = {mse:.4f}")

pred_lr = lr.predict(X_test)
pred_rf = rf.predict(X_test)
pred_gb = gb.predict(X_test)

evaluate(y_test, pred_lr, "Linear Regression")
evaluate(y_test, pred_rf, "Random Forest")
evaluate(y_test, pred_gb, "Gradient Boosting")

rmse_lr = np.sqrt(metrics.mean_squared_error(y_test, pred_lr))
rmse_rf = np.sqrt(metrics.mean_squared_error(y_test, pred_rf))
rmse_gb = np.sqrt(metrics.mean_squared_error(y_test, pred_gb))

r2_lr = metrics.r2_score(y_test, pred_lr)
r2_rf = metrics.r2_score(y_test, pred_rf)
r2_gb = metrics.r2_score(y_test, pred_gb)

mae_lr = metrics.mean_absolute_error(y_test, pred_lr)
mae_rf = metrics.mean_absolute_error(y_test, pred_rf)
mae_gb = metrics.mean_absolute_error(y_test, pred_gb)

mse_lr = metrics.mean_squared_error(y_test, pred_lr)
mse_rf = metrics.mean_squared_error(y_test, pred_rf)
mse_gb = metrics.mean_squared_error(y_test, pred_gb)

In [ ]:
#Visualize Metrics

import matplotlib.pyplot as plt
import numpy as np


models = ['Linear Regression', 'Random Forest', 'Gradient Boosting']
mae_values = [mae_lr, mae_rf, mae_gb]
mse_values = [mse_lr, mse_rf, mse_gb]
rmse_values = [rmse_lr, rmse_rf, rmse_gb]
r2_values = [r2_lr, r2_rf, r2_gb]

y_pos = np.arange(len(models))

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

axes[0].barh(y_pos, mae_values, align='center', color='skyblue')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(models)
axes[0].invert_yaxis()
axes[0].set_xlabel('MAE Value')
axes[0].set_title('Mean Absolute Error (MAE) by Model')
axes[0].grid(axis='x', linestyle='--', alpha=0.7)

axes[1].barh(y_pos, mse_values, align='center', color='lightcoral')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(models)
axes[1].invert_yaxis()
axes[1].set_xlabel('MSE Value')
axes[1].set_title('Mean Squared Error (MSE) by Model')
axes[1].grid(axis='x', linestyle='--', alpha=0.7)

axes[2].barh(y_pos, rmse_values, align='center', color='lightgreen')
axes[2].set_yticks(y_pos)
axes[2].set_yticklabels(models)
axes[2].invert_yaxis()
axes[2].set_xlabel('RMSE Value')
axes[2].set_title('Root Mean Squared Error (RMSE) by Model')
axes[2].grid(axis='x', linestyle='--', alpha=0.7)

axes[3].barh(y_pos, r2_values, align='center', color='gold')
axes[3].set_yticks(y_pos)
axes[3].set_yticklabels(models)
axes[3].invert_yaxis()
axes[3].set_xlabel('R² Value')
axes[3].set_title('R-squared (R²) by Model')
axes[3].grid(axis='x', linestyle='--', alpha=0.7)

fig.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

# STEP 8: CROSS VALIDATION

cv_scores = cross_val_score(rf, X_train, y_train, cv=10, scoring='r2')
print("Cross-validation R² (RF):", cv_scores.mean())

In [ ]:
# STEP 9: FEATURE IMPORTANCE


importance = pd.Series(rf.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\nTop 10 Important Features:\n")
print(importance.head(10))

In [ ]:
# STEP 10: VISUALIZATION

plt.figure(figsize=(12,5))
plt.plot(y_test.values, label="Actual")
plt.plot(pred_gb, label="Predicted")
plt.title("Global Gold Movement Prediction")
plt.legend()
plt.show()

In [ ]:
#10 VisualizationD

import matplotlib.pyplot as plt

#Linear Regression
plt.figure(figsize=(12,5))
plt.plot(y_test.values, label="Actual")
plt.plot(pred_lr, label="Predicted")
plt.title("Global Gold Movement Prediction (Linear Regression)")
plt.legend()
plt.show()

#Random Forest
plt.figure(figsize=(12,5))
plt.plot(y_test.values, label="Actual")
plt.plot(pred_rf, label="Predicted")
plt.title("Global Gold Movement Prediction (Random Forest)")
plt.legend()
plt.show()

#Gradient Boosting
plt.figure(figsize=(12,5))
plt.plot(y_test.values, label="Actual")
plt.plot(pred_gb, label="Predicted")
plt.title("Global Gold Movement Prediction (Gradient Boosting)")
plt.legend()
plt.show()